# Milestone 6 — Self-Supervised Multimodal Representation Learning

## 6.1 — SSL Dataset Preparation & Pair Availability Audit

### Objective

The objective of this audit was to characterize the naturally available multimodal data in the COde dataset before constructing the self-supervised learning pipeline.

Unlike the supervised baseline experiments, the SSL dataset is not restricted to complete multimodal visits. Each available modality pair is retained independently so that naturally missing modalities can be incorporated into the self-supervised training process.

### Dataset

The authoritative six-label patient-level dataset was used:

```text
results/six_label_patient_level_dataset/labeled_dataset.csv
```

The existing patient-level split was preserved and no new split was created.

Dataset size:

* 8,775 visits
* 4,800 patients

The patient-level split validation remained:

```text
Patients in multiple splits: 0
Status: PASS
```

### Modality Availability

| Modality / Pair  | Visits | Coverage |
| ---------------- | -----: | -------: |
| Image            |  8,772 |   99.97% |
| Radiograph       |  4,256 |   48.50% |
| Clinical Text    |  8,701 |   99.16% |
| Image–Text       |  8,698 |   99.12% |
| Image–Radiograph |  4,255 |   48.49% |
| Radiograph–Text  |  4,196 |   47.82% |
| Complete Triplet |  4,195 |   47.81% |

### Natural Modality Patterns

The observed visit-level modality patterns were:

| Pattern                   | Visits | Coverage |
| ------------------------- | -----: | -------: |
| Image + Text              |  4,503 |   51.32% |
| Image + Radiograph + Text |  4,195 |   47.81% |
| Image + Radiograph        |     60 |    0.68% |
| Image only                |     14 |    0.16% |
| Text only                 |      2 |    0.02% |
| Radiograph + Text         |      1 |    0.01% |

No visit was found without any available modality.

### SSL Pair Strategy

Based on the observed modality availability, all three cross-modal pairs will be retained:

```text
Image ↔ Text
Image ↔ Radiograph
Radiograph ↔ Text
```

A visit contributes only to the losses for which both required modalities are available.

For example:

```text
Image + Radiograph + Text
→ Image–Text loss
→ Image–Radiograph loss
→ Radiograph–Text loss

Image + Text
→ Image–Text loss

Image + Radiograph
→ Image–Radiograph loss

Radiograph + Text
→ Radiograph–Text loss
```

Therefore, missing modalities do not cause an otherwise usable visit to be discarded from SSL training.

### Observation

Image–Text pairs are available for almost the entire dataset, whereas radiograph-related pairs are available for approximately half of the visits. This distribution reflects the naturally missing radiograph characteristic of the COde dataset.

Because the Image–Text pair is substantially more frequent than the radiograph-related pairs, the relative contribution of different pair losses will need to be monitored during SSL training. Pair balancing or weighting may be considered if the first training experiments indicate that one objective dominates the optimization.

### Output Artifacts

The audit generated:

```text
results/ssl_dataset_audit/
├── audit_summary.json
├── modality_pattern_distribution.csv
├── pair_availability_by_split.csv
└── split_availability.csv
```

### Conclusion

The audit confirms that the COde dataset is suitable for a dynamic multimodal self-supervised learning setup. The dataset contains a large number of Image–Text pairs and substantial numbers of naturally occurring radiograph-related pairs, while preserving the original patient-level split without leakage.

The next step is to implement a dynamic SSL dataset that exposes only the modalities available for each visit and supports pair-specific contrastive objectives.


# 6.2 — Dynamic Multimodal SSL Dataset

## Objective

The supervised multimodal baseline used complete-case visits, requiring photographs, radiographs, and clinical text to be simultaneously available.

For self-supervised pretraining, this restriction is removed because the primary research objective is robustness to naturally missing modalities.

The SSL dataset therefore preserves all visits from the authoritative patient-level splits, regardless of modality availability.

## Dataset Design

Each sample corresponds to one dental visit and contains:

- Photographs — optional
- Radiographs — optional
- Clinical text — optional

Missing modalities are represented explicitly rather than causing the visit to be removed.

Example:

| Visit | Image | Radiograph | Text | Usable Pairs |
|---|---|---|---|---|
| A | ✓ | ✓ | ✓ | Image–Text, Image–Radiograph, Radiograph–Text |
| B | ✓ | ✗ | ✓ | Image–Text |
| C | ✓ | ✓ | ✗ | Image–Radiograph |
| D | ✗ | ✗ | ✓ | No multimodal pair |

Visits with no available modality pair remain in the dataset but contribute no contrastive loss.

## Text Input

The SSL text representation follows the same leakage-avoidance policy used in the supervised text baseline.

Included fields:

- `chief_complaint`
- `present_illness`
- `past_medical_record`
- `examination`

Excluded fields include `anomalies_en`, `diagnosis`, `treatment_plan`, `treatment_recommendations`, and `management`.

Labels are not required by the SSL dataset.

## Implementation

A dedicated dynamic SSL dataset was implemented:

`src/ssl/dataset.py`

and its batch collation function:

`src/ssl/collate.py`

The dataset does not perform complete-case filtering and preserves variable numbers of photographs and radiographs per visit.

Each sample exposes:

- `checkup_id`
- `patient_id`
- `images`
- `radiographs`
- `text`
- `has_image`
- `has_radiograph`
- `has_text`

## Sanity Check

The training split contains:

- 6,129 visits

The implementation was verified using a DataLoader with batch size 8.

The sanity check confirmed that visits with missing radiographs or photographs are retained and that modality availability is correctly represented at sample and batch level.

Therefore, the SSL dataset is suitable for dynamic contrastive learning where the available modality pairs can determine the loss contribution for each sample/batch.

### 6.3 — Multimodal Encoders

Three independent modality-specific encoders were implemented for self-supervised multimodal representation learning:

- **Photographs:** ImageNet-pretrained ResNet50 → 2048-dimensional representation
- **Radiographs:** Independent ImageNet-pretrained ResNet50 → 2048-dimensional representation
- **Clinical Text:** DistilBERT → 768-dimensional representation

The photograph and radiograph encoders do not share weights because the two image modalities have different visual characteristics and distributions.

At this stage, no classifier is used. The encoders only produce modality-specific representations. Projection heads will subsequently map these representations into a shared 128-dimensional embedding space for contrastive learning.

**Sanity check:**  
Image `(2, 3, 224, 224)` → `(2, 2048)`  
Radiograph `(2, 3, 224, 224)` → `(2, 2048)`  
Text `(2, 256)` → `(2, 768)`

**Status:** PASS

### 6.4 — Projection Head Design

To align representations from the three modalities in a shared embedding space, a separate projection head was added to each modality-specific encoder.

Architecture:

- Photograph: `2048 → 512 → 128`
- Radiograph: `2048 → 512 → 128`
- Clinical Text: `768 → 512 → 128`

Each projection head is a two-layer MLP with ReLU activation. The final 128-dimensional embedding is L2-normalized before contrastive learning.

This allows cosine similarity between representations from different modalities to be computed in the same embedding space.

The projection heads are modality-specific and do not share parameters.

**Sanity check:**

```text
Image       → (4, 2048) → (4, 128)
Radiograph  → (4, 2048) → (4, 128)
Text        → (4, 768)  → (4, 128)

# 6.5 — Dynamic Contrastive Learning

## Objective

The goal of this stage is to align the representations produced by the three
modalities of the COde dataset:

- Photograph
- Radiograph
- Clinical Text

Representations belonging to the same dental visit should be close in the
shared embedding space, while representations from different visits should be
far apart.

This is implemented using a symmetric InfoNCE / CLIP-style contrastive loss.

---

## Positive and Negative Pairs

For a given visit, representations from the same visit form positive pairs.

For example:

    Image_i ↔ Text_i

is a positive pair.

Other samples within the batch act as negative examples:

    Image_i ↔ Text_j,  i != j

Therefore, the contrastive objective uses the batch itself to construct
negative examples and does not require manually defined negative labels.

---

## Pair Types

The SSL framework supports three possible modality pairs:

    Image ↔ Text
    Image ↔ Radiograph
    Radiograph ↔ Text

Each pair is optimized using the same symmetric contrastive objective.

---

## Symmetric InfoNCE Loss

For two modality embedding matrices:

    Z_A ∈ R^(N × D)
    Z_B ∈ R^(N × D)

the embeddings are first L2-normalized.

The similarity matrix is then computed using cosine similarity:

    S_ij = (z_A_i · z_B_j) / τ

where:

- N is the number of valid pairs in the batch
- D is the embedding dimension
- τ is the temperature parameter

The diagonal elements correspond to positive pairs because sample i in
modality A belongs to the same visit as sample i in modality B.

The loss is computed in both directions:

    A → B
    B → A

and averaged:

    L_A↔B = 1/2 (L_A→B + L_B→A)

The temperature used in the current implementation is:

    τ = 0.07

---

## Dynamic Missing-Modality Handling

A central requirement of this thesis is robustness to naturally missing
modalities.

Therefore, the SSL objective does NOT require every visit to contain all
three modalities.

For each batch, modality-pair losses are calculated only when the required
modalities are available.

For example:

    Visit A:
        Image ✓
        Radiograph ✓
        Text ✓

    → Image-Text loss
    → Image-Radiograph loss
    → Radiograph-Text loss


    Visit B:
        Image ✓
        Radiograph ✗
        Text ✓

    → Image-Text loss only


    Visit C:
        Image ✓
        Radiograph ✓
        Text ✗

    → Image-Radiograph loss only

Thus, a missing radiograph does not cause the entire visit to be discarded.

This design allows the self-supervised representation learning stage to use
the naturally incomplete multimodal dataset rather than restricting training
to complete multimodal cases.

---

## Dynamic Loss Aggregation

If multiple modality pairs are available in a batch, their losses are averaged.

For example, if all three pairs are available:

    L_SSL =
        mean(
            L_Image-Text,
            L_Image-Radiograph,
            L_Radiograph-Text
        )

If only Image-Text is available:

    L_SSL = L_Image-Text

Therefore, the number of active objectives can change dynamically from batch
to batch.

---

## Implementation

The implementation is located at:

    src/ssl/contrastive.py

Main components:

    SymmetricContrastiveLoss
        ↓
    Symmetric InfoNCE / CLIP-style objective

    DynamicMultimodalContrastiveLoss
        ↓
    Dynamic selection and aggregation of modality-pair losses

Supported pairs:

    image_text
    image_radiograph
    radiograph_text

---

## Sanity Check

A synthetic batch with:

    Batch size = 8
    Embedding dimension = 128

was used to verify the implementation.

For a complete multimodal batch:

    Image-Text loss:          2.5425
    Image-Radiograph loss:    2.8882
    Radiograph-Text loss:     2.9594

    Total loss:               2.7967

All three modality streams produced valid gradients after backpropagation.

A second test simulated missing radiographs.

Only the Image-Text objective was active:

    Image-Text loss:           2.8590
    Total loss:                2.8590

The Image and Text representations received valid gradients, while the
missing Radiograph modality was correctly excluded from the objective.

This confirms that the dynamic contrastive loss behaves as intended under
naturally missing modalities.

---

## Design Rationale

The objective of this stage is not to directly optimize the final diagnostic
labels.

Instead, it learns modality-invariant and cross-modal representations before
supervised classification.

This provides the foundation for the later downstream experiment, where
the SSL-pretrained encoders will be compared against the baseline encoders.

The main hypothesis is that cross-modal representation learning can provide
more robust representations when one modality, particularly the radiograph,
is unavailable.

# Milestone 6.6 — SSL Training Pipeline

## Objective

The objective of this milestone was to implement a self-supervised multimodal pretraining pipeline for learning modality-aligned representations from the COde dental dataset.

Unlike supervised learning, where the model directly optimizes diagnostic labels, this stage aims to learn a shared representation space between available clinical modalities before downstream classification.

The designed pipeline focuses on robust multimodal representation learning under naturally missing modality conditions.

---

## Dataset and Modality Handling

The SSL pipeline was trained using the patient-level training split:

* Training visits: 6,129
* Patients: 3,360
* Modalities:

  * Intraoral photographs
  * Dental radiographs
  * Clinical text

The dataset was designed to preserve naturally missing modalities. Instead of removing incomplete visits, the pipeline dynamically determines available modality pairs during training.

This allows the model to learn from:

* Image–Text pairs
* Image–Radiograph pairs
* Radiograph–Text pairs

whenever the required modalities are available.

---

## Model Design

A dynamic multimodal contrastive learning framework was implemented.

Each modality is processed by a dedicated encoder:

* Image encoder:

  * CNN-based visual encoder

* Radiograph encoder:

  * Dedicated radiograph representation encoder

* Text encoder:

  * DistilBERT-based clinical text encoder

The modality-specific embeddings are projected into a shared latent space.

The training objective encourages semantically related modalities from the same visit to have similar embeddings while separating unrelated samples.

---

## Contrastive Learning Strategy

A dynamic pair-wise contrastive loss was used.

For each batch, available modality pairs are automatically selected using modality masks:

* Image ↔ Text
* Image ↔ Radiograph
* Radiograph ↔ Text

Missing modalities do not generate invalid pairs and are excluded from the loss computation.

This design matches the main research objective of the thesis: robust multimodal learning in the presence of naturally missing radiographs.

---

## Training Configuration

The SSL training configuration was:

* Epochs: 50
* Batch size: 16
* Optimizer: AdamW
* Learning rate: 1e-4
* Mixed precision training: Enabled
* Hardware:

  * NVIDIA RTX 3050 Laptop GPU (4GB VRAM)

The training pipeline completed successfully without memory overflow.

---

## Training Results

The contrastive loss showed a stable decreasing trend:

| Epoch | Loss   |
| ----- | ------ |
| 1     | 1.6916 |
| 10    | 0.3634 |
| 20    | 0.1743 |
| 30    | 0.1125 |
| 40    | 0.0933 |
| 45    | 0.0597 |
| 50    | 0.0739 |

The best checkpoint was obtained at:

* Best epoch: 45
* Best loss: 0.0597

Pair-wise losses at the best checkpoint indicated successful alignment across all modality combinations:

* Image–Text: 0.0847
* Image–Radiograph: 0.0256
* Radiograph–Text: 0.0688

---

## Output Artifacts

The SSL training stage generated:

```
results/
└── ssl_pretraining/
    └── full_dynamic/
        ├── best_ssl_model.pt
        └── history.json
```

The saved encoder checkpoint will be used in the next milestone for downstream representation evaluation and multimodal fusion experiments.

---

## Conclusion

The SSL training pipeline was successfully implemented. The decreasing contrastive loss and stable optimization across all modality pairs demonstrate that the model learned aligned multimodal representations while preserving robustness to naturally missing modalities.

This milestone provides the pretrained representation foundation for subsequent downstream classification and missing-modality robustness experiments.
